<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/04_TCN_CHL_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# STEP 31.9P-16I — TCN ARCHITECTURE DESIGN
# ================================================================

import os
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    GlobalAveragePooling1D,
    LayerNormalization,
    Activation,
    Add
)

# ================================================================
# PROJECT DIRECTORIES
# ================================================================

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

TCN_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "05_TCN"
)

ARCH_DIR = os.path.join(
    TCN_DIR,
    "architectures"
)

EVAL_DIR = os.path.join(
    TCN_DIR,
    "evaluation"
)

os.makedirs(TCN_DIR, exist_ok=True)
os.makedirs(ARCH_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print("=" * 70)
print("STEP 31.9P-16I")
print("TCN ARCHITECTURE DESIGN")
print("=" * 70)

print()
print("Project directory:")
print(PROJECT_DIR)

print()
print("TCN directory:")
print(TCN_DIR)

# ================================================================
# REPRODUCIBILITY
# ================================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print()
print("=" * 70)
print("REPRODUCIBILITY")
print("=" * 70)

print("Random seed:", SEED)

# ================================================================
# GPU VERIFICATION
# ================================================================

print()
print("=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

gpus = tf.config.list_physical_devices("GPU")

print("Number of GPUs detected:", len(gpus))

if len(gpus) > 0:
    for gpu in gpus:
        print("GPU:", gpu)
    print("✓ TensorFlow GPU available.")
else:
    print("⚠ No GPU detected.")
    print("Architecture design can continue on CPU.")

# ================================================================
# TCN CONFIGURATION
# ================================================================

SEQUENCE_LENGTH = 7
NUM_FEATURES = 69

CNN_FILTERS = 64
KERNEL_SIZE = 3

DILATIONS = [1, 2, 4]

DROPOUT_RATE = 0.20
DENSE_UNITS = 32

TARGET_NAME = "chlorophyll_a_log1p"

INPUT_SHAPE = (
    SEQUENCE_LENGTH,
    NUM_FEATURES
)

print()
print("=" * 70)
print("TCN CONFIGURATION")
print("=" * 70)

print("Sequence length :", SEQUENCE_LENGTH)
print("Number features  :", NUM_FEATURES)
print("Input shape      :", INPUT_SHAPE)
print("Filters          :", CNN_FILTERS)
print("Kernel size      :", KERNEL_SIZE)
print("Dilations        :", DILATIONS)
print("Dropout          :", DROPOUT_RATE)
print("Dense units      :", DENSE_UNITS)
print("Target           :", TARGET_NAME)

# ================================================================
# TCN BLOCK
# ================================================================

def tcn_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout_rate,
    block_name
):

    # Causal dilated convolution
    conv = Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        name=f"{block_name}_conv"
    )(x)

    conv = LayerNormalization(
        name=f"{block_name}_norm"
    )(conv)

    conv = Activation(
        "relu",
        name=f"{block_name}_relu"
    )(conv)

    conv = Dropout(
        dropout_rate,
        name=f"{block_name}_dropout"
    )(conv)

    # Residual connection
    if x.shape[-1] != filters:

        residual = Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            name=f"{block_name}_residual"
        )(x)

    else:
        residual = x

    output = Add(
        name=f"{block_name}_add"
    )([residual, conv])

    return output

# ================================================================
# BUILD TCN MODEL
# ================================================================

print()
print("=" * 70)
print("BUILDING TCN")
print("=" * 70)

inputs = Input(
    shape=INPUT_SHAPE,
    name="tcn_input"
)

x = inputs

for i, dilation in enumerate(DILATIONS, start=1):

    x = tcn_block(
        x=x,
        filters=CNN_FILTERS,
        kernel_size=KERNEL_SIZE,
        dilation_rate=dilation,
        dropout_rate=DROPOUT_RATE,
        block_name=f"tcn_block_{i}"
    )

x = GlobalAveragePooling1D(
    name="global_average_pooling"
)(x)

x = Dense(
    DENSE_UNITS,
    activation="relu",
    name="dense"
)(x)

x = Dropout(
    DROPOUT_RATE,
    name="final_dropout"
)(x)

outputs = Dense(
    1,
    activation="linear",
    name="output_chl_log1p"
)(x)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="TCN_CHL_Forecaster"
)

# ================================================================
# MODEL SUMMARY
# ================================================================

print()
print("=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

model.summary()

# ================================================================
# PARAMETER COUNT
# ================================================================

total_params = model.count_params()

trainable_params = np.sum([
    np.prod(variable.shape)
    for variable in model.trainable_variables
])

non_trainable_params = np.sum([
    np.prod(variable.shape)
    for variable in model.non_trainable_variables
])

print()
print("=" * 70)
print("MODEL PARAMETER COUNT")
print("=" * 70)

print("Total parameters      :", int(total_params))
print("Trainable parameters  :", int(trainable_params))
print("Non-trainable params  :", int(non_trainable_params))

# ================================================================
# OUTPUT SHAPE CHECK
# ================================================================

print()
print("=" * 70)
print("OUTPUT SHAPE CHECK")
print("=" * 70)

dummy_input = np.zeros(
    (1, SEQUENCE_LENGTH, NUM_FEATURES),
    dtype=np.float32
)

dummy_output = model.predict(
    dummy_input,
    verbose=0
)

print("Dummy input shape :", dummy_input.shape)
print("Dummy output shape:", dummy_output.shape)

assert dummy_output.shape == (1, 1)

print("✓ Output shape correctly configured as (1, 1).")

# ================================================================
# ARCHITECTURE VERIFICATION
# ================================================================

print()
print("=" * 70)
print("ARCHITECTURE VERIFICATION")
print("=" * 70)

layer_names = [
    layer.name
    for layer in model.layers
]

for name in layer_names:
    print(" -", name)

# Check expected TCN components
assert any(
    "tcn_block_1_conv" in name
    for name in layer_names
)

assert any(
    "tcn_block_2_conv" in name
    for name in layer_names
)

assert any(
    "tcn_block_3_conv" in name
    for name in layer_names
)

assert "output_chl_log1p" in layer_names

print()
print("✓ TCN blocks verified.")
print("✓ Dilated temporal convolutions configured.")
print("✓ Global temporal pooling configured.")
print("✓ Dense layer configured.")
print("✓ Single linear output configured.")

# ================================================================
# ARCHITECTURE CONFIGURATION
# ================================================================

architecture_config = {

    "experiment": "31.9P-16I",

    "model_name": "TCN_CHL_Forecaster",

    "sequence_length": SEQUENCE_LENGTH,

    "number_of_features": NUM_FEATURES,

    "input_shape": list(INPUT_SHAPE),

    "target": TARGET_NAME,

    "filters": CNN_FILTERS,

    "kernel_size": KERNEL_SIZE,

    "dilations": DILATIONS,

    "padding": "causal",

    "dropout": DROPOUT_RATE,

    "dense_units": DENSE_UNITS,

    "output_units": 1,

    "output_activation": "linear",

    "total_parameters": int(total_params),

    "trainable_parameters": int(trainable_params),

    "non_trainable_parameters": int(non_trainable_params),

    "training_performed": False,

    "validation_used": False,

    "test_data_used": False
}

architecture_json = os.path.join(
    ARCH_DIR,
    "tcn_architecture_summary.json"
)

with open(
    architecture_json,
    "w"
) as f:

    json.dump(
        architecture_config,
        f,
        indent=4
    )

# ================================================================
# SAVE UNTRAINED MODEL
# ================================================================

untrained_model_path = os.path.join(
    ARCH_DIR,
    "tcn_untrained.keras"
)

model.save(
    untrained_model_path
)

print()
print("=" * 70)
print("ARCHITECTURE SAVED")
print("=" * 70)

print("Architecture configuration:")
print(architecture_json)

print()
print("Untrained model:")
print(untrained_model_path)

# ================================================================
# SAFETY CHECK
# ================================================================

print()
print("=" * 70)
print("SAFETY CHECK")
print("=" * 70)

print("Training performed :", False)
print("Validation used    :", False)
print("Test data used     :", False)

print()
print("✓ No training performed.")
print("✓ No validation data used.")
print("✓ Test data was NOT loaded.")
print("✓ Existing RF model preserved.")
print("✓ Existing XGBoost model preserved.")
print("✓ Existing CNN-LSTM models preserved.")

# ================================================================
# COMPLETE
# ================================================================

print()
print("=" * 70)
print("STEP 31.9P-16I COMPLETE")
print("=" * 70)

print("✓ TCN architecture created.")
print("✓ 7-day sequence preserved.")
print("✓ 69 predictors preserved.")
print("✓ Causal temporal convolutions configured.")
print("✓ Dilations 1, 2, 4 configured.")
print("✓ Residual TCN blocks configured.")
print("✓ Dense layer configured.")
print("✓ Linear CHL output configured.")
print("✓ Parameter count recorded.")
print("✓ Architecture saved.")
print("✓ Untrained model saved.")
print("✓ Test data NOT used.")

STEP 31.9P-16I
TCN ARCHITECTURE DESIGN

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

TCN directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN

REPRODUCIBILITY
Random seed: 42

GPU VERIFICATION
Number of GPUs detected: 1
GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
✓ TensorFlow GPU available.

TCN CONFIGURATION
Sequence length : 7
Number features  : 69
Input shape      : (7, 69)
Filters          : 64
Kernel size      : 3
Dilations        : [1, 2, 4]
Dropout          : 0.2
Dense units      : 32
Target           : chlorophyll_a_log1p

BUILDING TCN

MODEL SUMMARY


Model: "TCN_CHL_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tcn_input           │ (None, 7, 69)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv    │ (None, 7, 64)     │     13,312 │ tcn_input[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_norm    │ (None, 7, 64)     │        128 │ tcn_block_1_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_relu    │ (None, 7, 64)     │          0 │ tcn_block_1_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_residu… │ (None, 7, 64)     │      4,480 │ tcn_input[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropout │ (None, 7, 64)     │          0 │ tcn_block_1_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_add     │ (None, 7, 64)     │          0 │ tcn_block_1_resi… │
│ (Add)               │                   │            │ tcn_block_1_drop… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv    │ (None, 7, 64)     │     12,352 │ tcn_block_1_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_norm    │ (None, 7, 64)     │        128 │ tcn_block_2_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_relu    │ (None, 7, 64)     │          0 │ tcn_block_2_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropout │ (None, 7, 64)     │          0 │ tcn_block_2_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_add     │ (None, 7, 64)     │          0 │ tcn_block_1_add[… │
│ (Add)               │                   │            │ tcn_block_2_drop… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv    │ (None, 7, 64)     │     12,352 │ tcn_block_2_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_norm    │ (None, 7, 64)     │        128 │ tcn_block_3_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_relu    │ (None, 7, 64)     │          0 │ tcn_block_3_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropout │ (None, 7, 64)     │          0 │ tcn_block_3_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_add     │ (None, 7, 64)     │          0 │ tcn_block_2_add[

 Total params: 44,993 (175.75 KB)

 Trainable params: 44,993 (175.75 KB)

 Non-trainable params: 0 (0.00 B)


MODEL PARAMETER COUNT
Total parameters      : 44993
Trainable parameters  : 44993
Non-trainable params  : 8

OUTPUT SHAPE CHECK
Dummy input shape : (1, 7, 69)
Dummy output shape: (1, 1)
✓ Output shape correctly configured as (1, 1).

ARCHITECTURE VERIFICATION
 - tcn_input
 - tcn_block_1_conv
 - tcn_block_1_norm
 - tcn_block_1_relu
 - tcn_block_1_residual
 - tcn_block_1_dropout
 - tcn_block_1_add
 - tcn_block_2_conv
 - tcn_block_2_norm
 - tcn_block_2_relu
 - tcn_block_2_dropout
 - tcn_block_2_add
 - tcn_block_3_conv
 - tcn_block_3_norm
 - tcn_block_3_relu
 - tcn_block_3_dropout
 - tcn_block_3_add
 - global_average_pooling
 - dense
 - final_dropout
 - output_chl_log1p

✓ TCN blocks verified.
✓ Dilated temporal convolutions configured.
✓ Global temporal pooling configured.
✓ Dense layer configured.
✓ Single linear output configured.

ARCHITECTURE SAVED
Architecture configuration:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/architectures/tcn_architecture_summary.jso

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================================================
# STEP 31.9P-16J — DISCOVER EXISTING CNN-LSTM SEQUENCE FILES
# ================================================================

import os

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

CNN_LSTM_DIR = os.path.join(
    PROJECT_DIR,
    "06_models",
    "03_CNN_LSTM"
)

CNN_LSTM_SEQ_DIR = os.path.join(
    CNN_LSTM_DIR,
    "sequences"
)

print("=" * 70)
print("STEP 31.9P-16J")
print("CNN-LSTM SEQUENCE FILE DISCOVERY")
print("=" * 70)

print()
print("CNN-LSTM directory:")
print(CNN_LSTM_DIR)

print()
print("Expected sequence directory:")
print(CNN_LSTM_SEQ_DIR)

print()
print("Directory exists:")
print(os.path.exists(CNN_LSTM_SEQ_DIR))

# ================================================================
# SEARCH ENTIRE CNN-LSTM DIRECTORY
# ================================================================

print()
print("=" * 70)
print("ALL NUMPY SEQUENCE FILES FOUND")
print("=" * 70)

numpy_files = []

if os.path.exists(CNN_LSTM_DIR):

    for root, dirs, files in os.walk(CNN_LSTM_DIR):

        for file in files:

            if file.lower().endswith(".npy"):

                full_path = os.path.join(
                    root,
                    file
                )

                numpy_files.append(full_path)

if len(numpy_files) == 0:

    print("No .npy files found.")

else:

    for path in sorted(numpy_files):

        print(path)

# ================================================================
# SEARCH FOR TRAINING / VALIDATION FILES
# ================================================================

print()
print("=" * 70)
print("TRAINING / VALIDATION CANDIDATES")
print("=" * 70)

keywords = [
    "train",
    "training",
    "validation",
    "val"
]

candidates = []

for path in numpy_files:

    filename = os.path.basename(path).lower()

    if any(
        keyword in filename
        for keyword in keywords
    ):

        candidates.append(path)

if len(candidates) == 0:

    print("No training/validation .npy candidates found.")

else:

    for path in sorted(candidates):

        print(path)

# ================================================================
# SEARCH FOR FILES CONTAINING X / Y
# ================================================================

print()
print("=" * 70)
print("X / Y SEQUENCE CANDIDATES")
print("=" * 70)

xy_candidates = []

for path in numpy_files:

    filename = os.path.basename(path).lower()

    if (
        "x" in filename
        or "y" in filename
    ):

        xy_candidates.append(path)

for path in sorted(xy_candidates):

    print(path)

# ================================================================
# DIRECTORY CONTENTS
# ================================================================

print()
print("=" * 70)
print("CNN-LSTM SEQUENCE DIRECTORY CONTENTS")
print("=" * 70)

if os.path.exists(CNN_LSTM_SEQ_DIR):

    contents = sorted(
        os.listdir(CNN_LSTM_SEQ_DIR)
    )

    if len(contents) == 0:

        print("Directory is empty.")

    else:

        for item in contents:

            item_path = os.path.join(
                CNN_LSTM_SEQ_DIR,
                item
            )

            if os.path.isfile(item_path):

                print(
                    "[FILE]  ",
                    item
                )

            else:

                print(
                    "[FOLDER]",
                    item
                )

else:

    print(
        "CNN-LSTM sequence directory does not exist."
    )

print()
print("=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

print()
print("⚠ DO NOT TRAIN THE TCN YET.")
print("⚠ DO NOT REGENERATE SEQUENCES YET.")
print("⚠ DO NOT USE TEST DATA.")

print()
print(
    "Send me the complete output from this cell."
)

STEP 31.9P-16J
CNN-LSTM SEQUENCE FILE DISCOVERY

CNN-LSTM directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM

Expected sequence directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/sequences

Directory exists:
False

ALL NUMPY SEQUENCE FILES FOUND
No .npy files found.

TRAINING / VALIDATION CANDIDATES
No training/validation .npy candidates found.

X / Y SEQUENCE CANDIDATES

CNN-LSTM SEQUENCE DIRECTORY CONTENTS
CNN-LSTM sequence directory does not exist.

DISCOVERY COMPLETE

⚠ DO NOT TRAIN THE TCN YET.
⚠ DO NOT REGENERATE SEQUENCES YET.
⚠ DO NOT USE TEST DATA.

Send me the complete output from this cell.


In [ ]:
# ================================================================
# STEP 31.9P-16J
# GLOBAL SEQUENCE DISCOVERY
# ================================================================

import os

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

print("=" * 70)
print("STEP 31.9P-16J")
print("GLOBAL SEQUENCE DISCOVERY")
print("=" * 70)

print()
print("Project directory:")
print(PROJECT_DIR)

print()
print("Project directory exists:")
print(os.path.exists(PROJECT_DIR))

# ================================================================
# SEARCH FOR ALL .NPY FILES
# ================================================================

print()
print("=" * 70)
print("ALL .NPY FILES IN PROJECT")
print("=" * 70)

npy_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if file.lower().endswith(".npy"):

            full_path = os.path.join(root, file)

            npy_files.append(full_path)

if len(npy_files) == 0:

    print("No .npy files found anywhere in the project.")

else:

    for path in sorted(npy_files):

        print(path)

print()
print("Total .npy files found:", len(npy_files))

# ================================================================
# SEARCH FOR SEQUENCE-RELATED FILES
# ================================================================

print()
print("=" * 70)
print("SEQUENCE-RELATED FILES")
print("=" * 70)

sequence_keywords = [
    "sequence",
    "cnn_lstm",
    "lstm",
    "training",
    "validation",
    "test",
    "X_",
    "y_"
]

sequence_candidates = []

for path in npy_files:

    filename = os.path.basename(path).lower()

    if any(
        keyword.lower() in filename
        for keyword in sequence_keywords
    ):

        sequence_candidates.append(path)

if len(sequence_candidates) == 0:

    print("No sequence-related .npy files found.")

else:

    for path in sorted(sequence_candidates):

        print(path)

print()
print(
    "Total sequence-related candidates:",
    len(sequence_candidates)
)

# ================================================================
# SEARCH FOR CNN-LSTM DIRECTORIES
# ================================================================

print()
print("=" * 70)
print("DIRECTORIES CONTAINING CNN-LSTM")
print("=" * 70)

cnn_lstm_dirs = []

for root, dirs, files in os.walk(PROJECT_DIR):

    if "cnn" in root.lower() or "lstm" in root.lower():

        cnn_lstm_dirs.append(root)

if len(cnn_lstm_dirs) == 0:

    print("No CNN-LSTM directories found.")

else:

    for path in sorted(set(cnn_lstm_dirs)):

        print(path)

# ================================================================
# SEARCH FOR FILES WITH EXPECTED SEQUENCE NAMES
# ================================================================

print()
print("=" * 70)
print("EXPECTED CNN-LSTM SEQUENCE NAME SEARCH")
print("=" * 70)

expected_keywords = [
    "cnn_lstm_x_training",
    "cnn_lstm_y_log_training",
    "cnn_lstm_x_validation",
    "cnn_lstm_y_log_validation",
    "cnn_lstm_x_test",
    "cnn_lstm_y_log_test"
]

expected_matches = []

for path in npy_files:

    filename = os.path.basename(path).lower()

    for keyword in expected_keywords:

        if keyword in filename:

            expected_matches.append(path)
            break

if len(expected_matches) == 0:

    print(
        "None of the expected CNN-LSTM sequence filenames "
        "were found."
    )

else:

    for path in sorted(set(expected_matches)):

        print(path)

# ================================================================
# SEARCH FOR POSSIBLE OTHER TEMPORAL SEQUENCES
# ================================================================

print()
print("=" * 70)
print("POSSIBLE TEMPORAL SEQUENCE FILES")
print("=" * 70)

temporal_candidates = []

for path in npy_files:

    filename = os.path.basename(path).lower()

    temporal_terms = [
        "x_train",
        "x_training",
        "x_val",
        "x_validation",
        "y_train",
        "y_training",
        "y_val",
        "y_validation",
        "sequence",
        "seq"
    ]

    if any(
        term in filename
        for term in temporal_terms
    ):

        temporal_candidates.append(path)

if len(temporal_candidates) == 0:

    print("No possible temporal sequence files found.")

else:

    for path in sorted(set(temporal_candidates)):

        print(path)

# ================================================================
# FINAL STATUS
# ================================================================

print()
print("=" * 70)
print("DISCOVERY STATUS")
print("=" * 70)

print()
print("✓ No files were modified.")
print("✓ No files were copied.")
print("✓ No models were modified.")
print("✓ No training performed.")
print("✓ No validation performed.")
print("✓ No test data used.")

print()
print("IMPORTANT:")
print("Do NOT regenerate sequences yet.")
print("Do NOT train the TCN yet.")

print()
print("=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

STEP 31.9P-16J
GLOBAL SEQUENCE DISCOVERY

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Project directory exists:
True

ALL .NPY FILES IN PROJECT
No .npy files found anywhere in the project.

Total .npy files found: 0

SEQUENCE-RELATED FILES
No sequence-related .npy files found.

Total sequence-related candidates: 0

DIRECTORIES CONTAINING CNN-LSTM
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/04_CNN_LSTM_Optimized
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/04_CNN_LSTM_Optimized/evaluation
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/04_CNN_LSTM_Optimized/v2
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/04_CNN_LSTM_Optimized/v2/evaluation

EXPECTED CNN-LSTM SEQUENCE NAME SEARCH
None of the expected CNN-LSTM sequence filenames were found.

POSSIBLE TEMPORAL SEQUENCE FILES
No possible temporal sequence files found.

DISCOVERY STATUS

✓ No files were modified.
✓ No files were copied.
✓ No models were modif

In [ ]:
# ================================================================
# STEP 31.9P-16J
# FIND SOURCE DATA AND CNN-LSTM SEQUENCE CONFIGURATION
# ================================================================

import os

PROJECT_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

print("=" * 70)
print("STEP 31.9P-16J")
print("SOURCE DATA / SEQUENCE CONFIGURATION DISCOVERY")
print("=" * 70)

# ================================================================
# SEARCH FOR CNN-LSTM NOTEBOOKS
# ================================================================

print()
print("=" * 70)
print("CNN-LSTM NOTEBOOKS")
print("=" * 70)

notebooks = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if file.lower().endswith(".ipynb"):

            if (
                "cnn" in file.lower()
                or "lstm" in file.lower()
            ):

                notebooks.append(
                    os.path.join(root, file)
                )

if len(notebooks) == 0:
    print("No CNN-LSTM notebooks found.")

else:

    for path in sorted(notebooks):
        print(path)

# ================================================================
# SEARCH FOR MODEL-READY DATA
# ================================================================

print()
print("=" * 70)
print("MODEL-READY DATA FILES")
print("=" * 70)

model_ready_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        filename = file.lower()

        if (
            "model_ready" in filename
            or "final_model_ready" in filename
            or "preprocess" in filename
        ):

            if file.lower().endswith(
                (".csv", ".parquet", ".pkl", ".joblib")
            ):

                model_ready_files.append(
                    os.path.join(root, file)
                )

if len(model_ready_files) == 0:

    print("No model-ready data candidates found.")

else:

    for path in sorted(model_ready_files):
        print(path)

# ================================================================
# SEARCH FOR SEQUENCE / SPLIT CONFIGURATION FILES
# ================================================================

print()
print("=" * 70)
print("SEQUENCE / SPLIT / CONFIGURATION FILES")
print("=" * 70)

configuration_files = []

keywords = [
    "sequence",
    "training_configuration",
    "preprocessing_summary",
    "split",
    "feature_columns",
    "feature_list",
    "configuration"
]

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        filename = file.lower()

        if any(
            keyword in filename
            for keyword in keywords
        ):

            if file.lower().endswith(
                (
                    ".csv",
                    ".json",
                    ".txt",
                    ".md",
                    ".ipynb"
                )
            ):

                configuration_files.append(
                    os.path.join(root, file)
                )

if len(configuration_files) == 0:

    print("No configuration candidates found.")

else:

    for path in sorted(
        set(configuration_files)
    ):

        print(path)

# ================================================================
# SEARCH FOR FINAL FEATURE LIST
# ================================================================

print()
print("=" * 70)
print("FEATURE LIST FILES")
print("=" * 70)

feature_files = []

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        filename = file.lower()

        if (
            "feature" in filename
            and file.lower().endswith(
                (".csv", ".json", ".txt")
            )
        ):

            feature_files.append(
                os.path.join(root, file)
            )

if len(feature_files) == 0:

    print("No feature-list files found.")

else:

    for path in sorted(set(feature_files)):
        print(path)

# ================================================================
# FINAL STATUS
# ================================================================

print()
print("=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

print()
print("✓ No files modified.")
print("✓ No models modified.")
print("✓ No sequences generated.")
print("✓ No training performed.")
print("✓ No validation performed.")
print("✓ No test data used.")

print()
print("IMPORTANT:")
print("Do NOT train the TCN yet.")
print("Do NOT create new sequences yet.")

print()
print(
    "Send me the complete output from this cell."
)

STEP 31.9P-16J
SOURCE DATA / SEQUENCE CONFIGURATION DISCOVERY

CNN-LSTM NOTEBOOKS
No CNN-LSTM notebooks found.

MODEL-READY DATA FILES
No model-ready data candidates found.

SEQUENCE / SPLIT / CONFIGURATION FILES
No configuration candidates found.

FEATURE LIST FILES
No feature-list files found.

DISCOVERY COMPLETE

✓ No files modified.
✓ No models modified.
✓ No sequences generated.
✓ No training performed.
✓ No validation performed.
✓ No test data used.

IMPORTANT:
Do NOT train the TCN yet.
Do NOT create new sequences yet.

Send me the complete output from this cell.


In [ ]:
# ======================================================================
# STEP 31.9P-16J — FIND ACTUAL PREPROCESSING FILES
# ======================================================================

import os

BASE_DIR = "/content/drive/MyDrive/Indhi_Marine_Research_Data"

print("=" * 70)
print("SEARCHING FOR ACTUAL MODEL-PREPROCESSING FILES")
print("=" * 70)

# ----------------------------------------------------------------------
# SEARCH ALL CSV FILES
# ----------------------------------------------------------------------

csv_files = []

for root, dirs, files in os.walk(BASE_DIR):

    for file in files:

        if file.lower().endswith(".csv"):

            csv_files.append(
                os.path.join(root, file)
            )

print()
print("Total CSV files found:", len(csv_files))

# ----------------------------------------------------------------------
# SEARCH FOR EXACT EXPECTED FILES
# ----------------------------------------------------------------------

expected_names = {
    "final_model_feature_columns.csv",
    "final_model_ready_train.csv",
    "final_model_ready_validation.csv",
    "final_model_ready_test.csv",
}

print()
print("=" * 70)
print("EXACT EXPECTED FILE SEARCH")
print("=" * 70)

for expected in sorted(expected_names):

    matches = [
        path
        for path in csv_files
        if os.path.basename(path).lower() == expected.lower()
    ]

    print()
    print(expected)

    if matches:

        for path in matches:
            print("FOUND:")
            print(path)

    else:

        print("NOT FOUND")

# ----------------------------------------------------------------------
# SEARCH RELATED FILENAMES
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("MODEL-READY / FEATURE-RELATED CSV FILES")
print("=" * 70)

keywords = [
    "model_ready",
    "feature_columns",
    "feature_list",
    "preprocessing",
    "train",
    "validation",
    "test"
]

related_files = []

for path in csv_files:

    filename = os.path.basename(path).lower()

    if any(
        keyword in filename
        for keyword in keywords
    ):

        related_files.append(path)

for path in sorted(set(related_files)):

    print(path)

# ----------------------------------------------------------------------
# SHOW 05_MODEL_PREPROCESSING DIRECTORY
# ----------------------------------------------------------------------

PREPROCESS_DIR = os.path.join(
    BASE_DIR,
    "05_model_preprocessing"
)

print()
print("=" * 70)
print("05_MODEL_PREPROCESSING DIRECTORY")
print("=" * 70)

print()
print("Directory:")
print(PREPROCESS_DIR)

print()
print("Exists:")
print(os.path.exists(PREPROCESS_DIR))

if os.path.exists(PREPROCESS_DIR):

    contents = sorted(
        os.listdir(PREPROCESS_DIR)
    )

    print()
    print("Contents:")

    for item in contents:

        item_path = os.path.join(
            PREPROCESS_DIR,
            item
        )

        if os.path.isfile(item_path):

            print("[FILE]  ", item)

        else:

            print("[FOLDER]", item)

# ----------------------------------------------------------------------
# FINAL SAFETY
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("SAFETY STATUS")
print("=" * 70)

print("Files modified: False")
print("Models modified: False")
print("Sequences generated: False")
print("Training performed: False")
print("Validation performed: False")
print("Test data loaded: False")

print()
print("✓ Diagnostic search only.")
print("✓ No files changed.")
print("✓ No test data loaded.")
print()
print("Do NOT train TCN yet.")

SEARCHING FOR ACTUAL MODEL-PREPROCESSING FILES

Total CSV files found: 175

EXACT EXPECTED FILE SEARCH

final_model_feature_columns.csv
FOUND:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

final_model_ready_test.csv
FOUND:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

final_model_ready_train.csv
FOUND:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv

final_model_ready_validation.csv
FOUND:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv

MODEL-READY / FEATURE-RELATED CSV FILES
/content/drive/MyDrive/Indhi_Marine_Research_Data/04_feature_engineering/final_modelling_feature_list.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/04_feature_engineering/temporal_train_validation_test_split_report.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_prepro

In [ ]:
# ======================================================================
# STEP 31.9P-16J
# TCN TRAINING + VALIDATION SEQUENCE GENERATION
#
# SAME METHODOLOGY AS ORIGINAL CNN-LSTM
#
# 7 historical daily observations
# 1-day-ahead forecast
# 69 predictors
# Same model-ready train/validation datasets
# TEST DATA NOT USED
# ======================================================================

import os
import gc
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path

print("=" * 70)
print("STEP 31.9P-16J")
print("TCN TRAINING + VALIDATION SEQUENCE GENERATION")
print("=" * 70)

# ======================================================================
# 1. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESS_DIR = (
    PROJECT_DIR /
    "05_model_preprocessing"
)

TCN_DIR = (
    PROJECT_DIR /
    "06_models" /
    "05_TCN"
)

SEQUENCE_DIR = (
    TCN_DIR /
    "sequences"
)

EVALUATION_DIR = (
    TCN_DIR /
    "evaluation"
)

SEQUENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("Project directory:")
print(PROJECT_DIR)

print()
print("TCN directory:")
print(TCN_DIR)

print()
print("Sequence directory:")
print(SEQUENCE_DIR)

# ======================================================================
# 2. CONFIGURATION
# ======================================================================

SEQUENCE_LENGTH = 7
FORECAST_HORIZON = 1

TARGET_LOG = "chlorophyll_a_log1p"
TARGET_ORIGINAL = "chlorophyll_a"

LATITUDE_COLUMN = "latitude"
LONGITUDE_COLUMN = "longitude"
DATE_COLUMN = "date"

FEATURE_FILE = (
    PREPROCESS_DIR /
    "final_model_feature_columns.csv"
)

TRAIN_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_train.csv"
)

VALIDATION_FILE = (
    PREPROCESS_DIR /
    "final_model_ready_validation.csv"
)

print()
print("=" * 70)
print("SEQUENCE CONFIGURATION")
print("=" * 70)

print(
    "Sequence length:",
    SEQUENCE_LENGTH,
    "days"
)

print(
    "Forecast horizon:",
    FORECAST_HORIZON,
    "day"
)

print(
    "Target:",
    TARGET_LOG
)

print(
    "Original target:",
    TARGET_ORIGINAL
)

print(
    "Predictors:",
    69
)

print(
    "Test data:",
    "NOT LOADED"
)

# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

print()
print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {
    "Feature list": FEATURE_FILE,
    "Training data": TRAIN_FILE,
    "Validation data": VALIDATION_FILE
}

for name, path in required_files.items():

    print()
    print(name + ":")
    print(path)

    print(
        "Exists:",
        path.exists()
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print()
print("✓ All required source files found.")

# ======================================================================
# 4. LOAD FEATURE LIST
# ======================================================================

print()
print("=" * 70)
print("LOADING FINAL FEATURE LIST")
print("=" * 70)

feature_df = pd.read_csv(
    FEATURE_FILE
)

print()
print("Feature-list columns:")
print(
    feature_df.columns.tolist()
)

# ----------------------------------------------------------------------
# Identify feature column
# ----------------------------------------------------------------------

possible_feature_columns = [
    "feature",
    "feature_name",
    "Feature",
    "Feature_Name"
]

feature_column = None

for column in possible_feature_columns:

    if column in feature_df.columns:

        feature_column = column
        break

if feature_column is None:

    raise ValueError(
        "Could not identify the feature-name column "
        "in final_model_feature_columns.csv"
    )

FEATURES = (
    feature_df[feature_column]
    .dropna()
    .astype(str)
    .tolist()
)

FEATURES = list(
    dict.fromkeys(FEATURES)
)

print()
print("Feature column:")
print(feature_column)

print()
print("Number of selected features:")
print(len(FEATURES))

if len(FEATURES) != 69:

    raise ValueError(
        f"Expected 69 features, found {len(FEATURES)}"
    )

print()
print("✓ Exactly 69 predictors confirmed.")

# ======================================================================
# 5. LOAD TRAINING DATA
# ======================================================================

print()
print("=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)

train_df = pd.read_csv(
    TRAIN_FILE
)

print()
print("Training shape:")
print(train_df.shape)

# ======================================================================
# 6. LOAD VALIDATION DATA
# ======================================================================

print()
print("=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

validation_df = pd.read_csv(
    VALIDATION_FILE
)

print()
print("Validation shape:")
print(validation_df.shape)

# ======================================================================
# 7. REQUIRED COLUMN CHECK
# ======================================================================

required_columns = (
    FEATURES
    + [
        TARGET_LOG,
        TARGET_ORIGINAL,
        DATE_COLUMN,
        LATITUDE_COLUMN,
        LONGITUDE_COLUMN
    ]
)

print()
print("=" * 70)
print("COLUMN EXISTENCE CHECK")
print("=" * 70)

for name, df in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    missing = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    print()
    print(name + " missing columns:")
    print(len(missing))

    if missing:

        print(missing)

        raise ValueError(
            f"{name} is missing required columns."
        )

print()
print("✓ All required columns available.")

# ======================================================================
# 8. DATE CONVERSION
# ======================================================================

train_df[DATE_COLUMN] = pd.to_datetime(
    train_df[DATE_COLUMN]
)

validation_df[DATE_COLUMN] = pd.to_datetime(
    validation_df[DATE_COLUMN]
)

print()
print("✓ Dates converted to datetime.")

# ======================================================================
# 9. DUPLICATE CHECK
# ======================================================================

print()
print("=" * 70)
print("LOCATION-DATE DUPLICATE CHECK")
print("=" * 70)

for name, df in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    duplicates = (
        df.duplicated(
            subset=[
                LATITUDE_COLUMN,
                LONGITUDE_COLUMN,
                DATE_COLUMN
            ]
        )
        .sum()
    )

    print()
    print(name)
    print(
        "Duplicate location-date rows:",
        duplicates
    )

    if duplicates != 0:

        raise ValueError(
            f"{name} contains duplicate "
            "location-date observations."
        )

print()
print(
    "✓ No duplicate location-date observations."
)

# ======================================================================
# 10. SAFETY CHECK
# ======================================================================

print()
print("=" * 70)
print("SOURCE DATA SAFETY CHECK")
print("=" * 70)

for name, df in [
    ("Training", train_df),
    ("Validation", validation_df)
]:

    predictor_nan = (
        df[FEATURES]
        .isna()
        .sum()
        .sum()
    )

    target_log_nan = (
        df[TARGET_LOG]
        .isna()
        .sum()
    )

    target_original_nan = (
        df[TARGET_ORIGINAL]
        .isna()
        .sum()
    )

    predictor_inf = np.isinf(
        df[FEATURES]
        .to_numpy(
            dtype=np.float32
        )
    ).sum()

    print()
    print(name)

    print(
        "Predictor NaN:",
        predictor_nan
    )

    print(
        "Predictor infinite:",
        predictor_inf
    )

    print(
        "Log target NaN:",
        target_log_nan
    )

    print(
        "Original target NaN:",
        target_original_nan
    )

    if predictor_nan != 0:
        raise ValueError(
            f"{name} contains predictor NaN values."
        )

    if predictor_inf != 0:
        raise ValueError(
            f"{name} contains infinite predictor values."
        )

    if target_log_nan != 0:
        raise ValueError(
            f"{name} contains log target NaN values."
        )

    if target_original_nan != 0:
        raise ValueError(
            f"{name} contains original target NaN values."
        )

print()
print(
    "✓ Training and validation data passed safety checks."
)

# ======================================================================
# 11. SEQUENCE GENERATION FUNCTION
# ======================================================================

def generate_sequences(
    df,
    split_name
):

    print()
    print("=" * 70)
    print(
        f"GENERATING {split_name.upper()} TCN SEQUENCES"
    )
    print("=" * 70)

    start_time = time.time()

    # --------------------------------------------------------------
    # Sort by spatial location and date
    # --------------------------------------------------------------

    df = (
        df
        .sort_values(
            [
                LATITUDE_COLUMN,
                LONGITUDE_COLUMN,
                DATE_COLUMN
            ]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------------
    # Count valid sequences first
    # --------------------------------------------------------------

    valid_count = 0

    for (
        lat,
        lon
    ), group in df.groupby(
        [
            LATITUDE_COLUMN,
            LONGITUDE_COLUMN
        ],
        sort=False
    ):

        group = (
            group
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )

        dates = (
            group[DATE_COLUMN]
            .to_numpy()
            .astype("datetime64[D]")
        )

        n = len(group)

        if n < (
            SEQUENCE_LENGTH +
            FORECAST_HORIZON
        ):

            continue

        gaps = (
            dates[1:] -
            dates[:-1]
        ).astype(
            "timedelta64[D]"
        ).astype(int)

        for start in range(
            0,
            n -
            (
                SEQUENCE_LENGTH +
                FORECAST_HORIZON
            )
            + 1
        ):

            target_index = (
                start +
                SEQUENCE_LENGTH +
                FORECAST_HORIZON -
                1
            )

            required_gaps = gaps[
                start:
                target_index
            ]

            if np.all(
                required_gaps == 1
            ):

                valid_count += 1

    print()
    print(
        "Valid sequences:",
        valid_count
    )

    if valid_count == 0:

        raise ValueError(
            f"No valid {split_name} sequences found."
        )

    # --------------------------------------------------------------
    # Allocate arrays
    # --------------------------------------------------------------

    X = np.empty(
        (
            valid_count,
            SEQUENCE_LENGTH,
            len(FEATURES)
        ),
        dtype=np.float32
    )

    y_log = np.empty(
        valid_count,
        dtype=np.float32
    )

    y_original = np.empty(
        valid_count,
        dtype=np.float32
    )

    metadata_date = np.empty(
        valid_count,
        dtype="datetime64[D]"
    )

    metadata_latitude = np.empty(
        valid_count,
        dtype=np.float32
    )

    metadata_longitude = np.empty(
        valid_count,
        dtype=np.float32
    )

    # --------------------------------------------------------------
    # Generate sequences
    # --------------------------------------------------------------

    sequence_index = 0

    for (
        lat,
        lon
    ), group in df.groupby(
        [
            LATITUDE_COLUMN,
            LONGITUDE_COLUMN
        ],
        sort=False
    ):

        group = (
            group
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )

        dates = (
            group[DATE_COLUMN]
            .to_numpy()
            .astype("datetime64[D]")
        )

        feature_values = (
            group[FEATURES]
            .to_numpy(
                dtype=np.float32
            )
        )

        target_log_values = (
            group[TARGET_LOG]
            .to_numpy(
                dtype=np.float32
            )
        )

        target_original_values = (
            group[TARGET_ORIGINAL]
            .to_numpy(
                dtype=np.float32
            )
        )

        n = len(group)

        if n < (
            SEQUENCE_LENGTH +
            FORECAST_HORIZON
        ):

            continue

        gaps = (
            dates[1:] -
            dates[:-1]
        ).astype(
            "timedelta64[D]"
        ).astype(int)

        for start in range(
            0,
            n -
            (
                SEQUENCE_LENGTH +
                FORECAST_HORIZON
            )
            + 1
        ):

            input_end = (
                start +
                SEQUENCE_LENGTH
            )

            target_index = (
                input_end +
                FORECAST_HORIZON -
                1
            )

            required_gaps = gaps[
                start:
                target_index
            ]

            # ------------------------------------------------------
            # Only accept consecutive daily observations
            # ------------------------------------------------------

            if not np.all(
                required_gaps == 1
            ):

                continue

            # ------------------------------------------------------
            # Input sequence
            # ------------------------------------------------------

            X[
                sequence_index
            ] = feature_values[
                start:input_end
            ]

            # ------------------------------------------------------
            # Target
            # ------------------------------------------------------

            y_log[
                sequence_index
            ] = target_log_values[
                target_index
            ]

            y_original[
                sequence_index
            ] = target_original_values[
                target_index
            ]

            # ------------------------------------------------------
            # Metadata
            # ------------------------------------------------------

            metadata_date[
                sequence_index
            ] = dates[
                target_index
            ]

            metadata_latitude[
                sequence_index
            ] = lat

            metadata_longitude[
                sequence_index
            ] = lon

            sequence_index += 1

    # ==================================================================
    # FINAL COUNT CHECK
    # ==================================================================

    if sequence_index != valid_count:

        raise ValueError(
            "Generated sequence count does not match "
            "the expected sequence count."
        )

    print()
    print(
        "✓ All expected sequences generated."
    )

    # ==================================================================
    # SHAPE CHECK
    # ==================================================================

    print()
    print("X shape:")
    print(X.shape)

    print()
    print("Log target shape:")
    print(y_log.shape)

    print()
    print("Original target shape:")
    print(y_original.shape)

    expected_shape = (
        SEQUENCE_LENGTH,
        len(FEATURES)
    )

    if X.shape[1:] != expected_shape:

        raise ValueError(
            f"Unexpected sequence shape: "
            f"{X.shape[1:]}"
        )

    # ==================================================================
    # SAFETY CHECK
    # ==================================================================

    print()
    print("X NaN:")
    print(np.isnan(X).sum())

    print(
        "X infinite:"
    )
    print(
        np.isinf(X).sum()
    )

    print(
        "y log NaN:"
    )
    print(
        np.isnan(y_log).sum()
    )

    print(
        "y original NaN:"
    )
    print(
        np.isnan(y_original).sum()
    )

    if np.isnan(X).any():
        raise ValueError("NaN detected in X.")

    if np.isinf(X).any():
        raise ValueError(
            "Infinite values detected in X."
        )

    if np.isnan(y_log).any():
        raise ValueError(
            "NaN detected in y_log."
        )

    if np.isnan(y_original).any():
        raise ValueError(
            "NaN detected in y_original."
        )

    print()
    print(
        "✓ Sequence safety checks passed."
    )

    # ==================================================================
    # TARGET CONSISTENCY CHECK
    # ==================================================================

    reconstructed_original = np.expm1(
        y_log
    )

    max_difference = np.max(
        np.abs(
            reconstructed_original -
            y_original
        )
    )

    print()
    print(
        "Maximum difference between:"
    )

    print(
        "expm1(y_log)"
    )

    print(
        "and stored original target:"
    )

    print(
        float(max_difference)
    )

    if max_difference > 1e-4:

        raise ValueError(
            "Log and original targets are inconsistent."
        )

    print(
        "✓ Log/original target consistency verified."
    )

    # ==================================================================
    # METADATA
    # ==================================================================

    metadata_df = pd.DataFrame(
        {
            "target_date":
                pd.to_datetime(
                    metadata_date
                ),

            "latitude":
                metadata_latitude,

            "longitude":
                metadata_longitude
        }
    )

    # ==================================================================
    # SAVE
    # ==================================================================

    split_lower = split_name.lower()

    X_path = (
        SEQUENCE_DIR /
        f"tcn_X_{split_lower}.npy"
    )

    y_log_path = (
        SEQUENCE_DIR /
        f"tcn_y_log_{split_lower}.npy"
    )

    y_original_path = (
        SEQUENCE_DIR /
        f"tcn_y_original_{split_lower}.npy"
    )

    metadata_path = (
        SEQUENCE_DIR /
        f"tcn_metadata_{split_lower}.csv"
    )

    print()
    print("=" * 70)
    print(
        f"SAVING {split_name.upper()} TCN SEQUENCES"
    )
    print("=" * 70)

    np.save(
        X_path,
        X,
        allow_pickle=False
    )

    np.save(
        y_log_path,
        y_log,
        allow_pickle=False
    )

    np.save(
        y_original_path,
        y_original,
        allow_pickle=False
    )

    metadata_df.to_csv(
        metadata_path,
        index=False
    )

    print()
    print("X:")
    print(X_path)

    print()
    print("Log target:")
    print(y_log_path)

    print()
    print("Original target:")
    print(y_original_path)

    print()
    print("Metadata:")
    print(metadata_path)

    # ==================================================================
    # SAMPLE METADATA
    # ==================================================================

    print()
    print(
        f"First 5 {split_name.lower()} metadata rows:"
    )

    print(
        metadata_df.head().to_string(
            index=False
        )
    )

    elapsed = (
        time.time() -
        start_time
    )

    print()
    print(
        "Sequence generation time:",
        f"{elapsed:.2f} seconds"
    )

    # ==================================================================
    # RELEASE MEMORY
    # ==================================================================

    result = {
        "split": split_name,
        "sequence_count": int(valid_count),
        "X_path": str(X_path),
        "y_log_path": str(y_log_path),
        "y_original_path": str(y_original_path),
        "metadata_path": str(metadata_path)
    }

    del X
    del y_log
    del y_original
    del metadata_df

    gc.collect()

    return result


# ======================================================================
# 12. GENERATE TRAINING SEQUENCES
# ======================================================================

train_result = generate_sequences(
    train_df,
    "training"
)

del train_df
gc.collect()

# ======================================================================
# 13. GENERATE VALIDATION SEQUENCES
# ======================================================================

validation_result = generate_sequences(
    validation_df,
    "validation"
)

del validation_df
gc.collect()

# ======================================================================
# 14. SAVE CONFIGURATION
# ======================================================================

sequence_configuration = {

    "model": "TCN",

    "sequence_length_days":
        SEQUENCE_LENGTH,

    "forecast_horizon_days":
        FORECAST_HORIZON,

    "number_of_features":
        len(FEATURES),

    "features":
        FEATURES,

    "target":
        TARGET_LOG,

    "original_target":
        TARGET_ORIGINAL,

    "sequence_rule":
        (
            "Seven consecutive daily observations "
            "from the same latitude/longitude location "
            "are used to predict CHL one day ahead."
        ),

    "continuity_rule":
        (
            "Every required observation must be exactly "
            "one day apart."
        ),

    "split_rule":
        (
            "Training and validation sequences are "
            "generated independently within their "
            "respective chronological datasets."
        ),

    "training_sequences":
        train_result["sequence_count"],

    "validation_sequences":
        validation_result["sequence_count"],

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "test_sequences_generated":
        False,

    "methodology":
        (
            "Same sequence construction methodology "
            "as the original CNN-LSTM experiment."
        )
}

configuration_path = (
    EVALUATION_DIR /
    "tcn_sequence_configuration.json"
)

with open(
    configuration_path,
    "w"
) as file:

    json.dump(
        sequence_configuration,
        file,
        indent=4
    )

# ======================================================================
# 15. FINAL FILE CHECK
# ======================================================================

print()
print("=" * 70)
print("FINAL TCN SEQUENCE FILE CHECK")
print("=" * 70)

required_sequence_files = [

    train_result["X_path"],
    train_result["y_log_path"],
    train_result["y_original_path"],
    train_result["metadata_path"],

    validation_result["X_path"],
    validation_result["y_log_path"],
    validation_result["y_original_path"],
    validation_result["metadata_path"]
]

for path in required_sequence_files:

    exists = Path(path).exists()

    print()
    print(
        Path(path).name,
        "Exists:",
        exists
    )

    if not exists:

        raise FileNotFoundError(
            f"Sequence file was not created: {path}"
        )

# ======================================================================
# 16. TEST DATA SAFETY
# ======================================================================

print()
print("=" * 70)
print("TEST DATA SAFETY CHECK")
print("=" * 70)

print(
    "Test dataset loaded:",
    False
)

print(
    "Test sequences generated:",
    False
)

print(
    "Test predictions generated:",
    False
)

print(
    "Test data used:",
    False
)

print()
print(
    "✓ Test data was NOT loaded."
)

print(
    "✓ Test data was NOT used."
)

print(
    "✓ No test sequences generated."
)

# ======================================================================
# 17. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("STEP 31.9P-16J COMPLETE")
print("=" * 70)

print()
print("✓ Training sequences generated.")

print(
    "✓ Validation sequences generated."
)

print(
    "✓ 7-day temporal structure preserved."
)

print(
    "✓ 69 predictors preserved."
)

print(
    "✓ 1-day forecast horizon preserved."
)

print(
    "✓ Spatial locations handled separately."
)

print(
    "✓ Daily continuity requirement enforced."
)

print(
    "✓ Log/original targets verified."
)

print(
    "✓ Sequence files saved."
)

print(
    "✓ Configuration saved."
)

print(
    "✓ Test data NOT loaded."
)

print(
    "✓ Test data NOT used."
)

print()
print(
    "Training sequences:",
    train_result["sequence_count"]
)

print(
    "Validation sequences:",
    validation_result["sequence_count"]
)

print()
print(
    "Configuration:"
)

print(
    configuration_path
)

print()
print(
    "Sequence directory:"
)

print(
    SEQUENCE_DIR
)

STEP 31.9P-16J
TCN TRAINING + VALIDATION SEQUENCE GENERATION

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

TCN directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN

Sequence directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/sequences

SEQUENCE CONFIGURATION
Sequence length: 7 days
Forecast horizon: 1 day
Target: chlorophyll_a_log1p
Original target: chlorophyll_a
Predictors: 69
Test data: NOT LOADED

REQUIRED FILE CHECK

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Exists: True

Training data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_train.csv
Exists: True

Validation data:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_validation.csv
Exists: True

✓ All required source files found.

LOADING FINAL FEATURE LIST

Feature-list columns:
['feature_in

In [ ]:
# ======================================================================
# STEP 31.9P-16K
# TCN TRAINING
#
# IMPORTANT:
# - Uses ONLY training and validation sequences
# - Test data is NOT loaded
# - Existing models are NOT modified
# - TCN architecture from 31.9P-16I is preserved
# ======================================================================

import os
import gc
import json
import random
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("=" * 70)
print("STEP 31.9P-16K")
print("TCN TRAINING")
print("=" * 70)

# ======================================================================
# 1. REPRODUCIBILITY
# ======================================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print()
print("=" * 70)
print("REPRODUCIBILITY")
print("=" * 70)

print("Random seed:", SEED)

# ======================================================================
# 2. GPU VERIFICATION
# ======================================================================

print()
print("=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

gpus = tf.config.list_physical_devices("GPU")

print(
    "Number of GPUs detected:",
    len(gpus)
)

if len(gpus) > 0:

    for gpu in gpus:
        print("GPU:", gpu)

    print()
    print("✓ TensorFlow GPU available.")

else:

    print()
    print("⚠ No GPU detected.")
    print("Training will use CPU.")

# ======================================================================
# 3. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

TCN_DIR = (
    PROJECT_DIR /
    "06_models" /
    "05_TCN"
)

SEQUENCE_DIR = (
    TCN_DIR /
    "sequences"
)

MODEL_DIR = (
    TCN_DIR /
    "models"
)

EVALUATION_DIR = (
    TCN_DIR /
    "evaluation"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print()
print("TCN directory:")
print(TCN_DIR)

print()
print("Sequence directory:")
print(SEQUENCE_DIR)

print()
print("Model directory:")
print(MODEL_DIR)

print()
print("Evaluation directory:")
print(EVALUATION_DIR)

# ======================================================================
# 4. TRAINING CONFIGURATION
# ======================================================================

SEQUENCE_LENGTH = 7
NUMBER_FEATURES = 69

TARGET = "chlorophyll_a_log1p"

BATCH_SIZE = 256
MAX_EPOCHS = 50

LEARNING_RATE = 0.001

HUBER_DELTA = 1.0

EARLY_STOPPING_PATIENCE = 8

REDUCE_LR_PATIENCE = 4
REDUCE_LR_FACTOR = 0.5

MIN_LR = 1e-6

print()
print("=" * 70)
print("TCN TRAINING CONFIGURATION")
print("=" * 70)

print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", NUMBER_FEATURES)
print("Input shape:", (SEQUENCE_LENGTH, NUMBER_FEATURES))

print("Target:", TARGET)

print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)

print("Learning rate:", LEARNING_RATE)

print("Loss: Huber")
print("Huber delta:", HUBER_DELTA)

print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "ReduceLR patience:",
    REDUCE_LR_PATIENCE
)

print(
    "ReduceLR factor:",
    REDUCE_LR_FACTOR
)

print("Minimum learning rate:", MIN_LR)

# ======================================================================
# 5. REQUIRED FILES
# ======================================================================

X_TRAIN_PATH = (
    SEQUENCE_DIR /
    "tcn_X_training.npy"
)

Y_TRAIN_PATH = (
    SEQUENCE_DIR /
    "tcn_y_log_training.npy"
)

X_VALIDATION_PATH = (
    SEQUENCE_DIR /
    "tcn_X_validation.npy"
)

Y_VALIDATION_PATH = (
    SEQUENCE_DIR /
    "tcn_y_log_validation.npy"
)

ARCHITECTURE_PATH = (
    TCN_DIR /
    "architectures" /
    "tcn_untrained.keras"
)

print()
print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {

    "Training X":
        X_TRAIN_PATH,

    "Training y":
        Y_TRAIN_PATH,

    "Validation X":
        X_VALIDATION_PATH,

    "Validation y":
        Y_VALIDATION_PATH,

    "TCN architecture":
        ARCHITECTURE_PATH
}

for name, path in required_files.items():

    print()
    print(name + ":")
    print(path)

    print(
        "Exists:",
        path.exists()
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print()
print("✓ All required training files found.")

# ======================================================================
# 6. LOAD TRAINING SEQUENCES
# ======================================================================

print()
print("=" * 70)
print("LOADING TRAINING SEQUENCES")
print("=" * 70)

X_train = np.load(
    X_TRAIN_PATH,
    allow_pickle=False
)

y_train = np.load(
    Y_TRAIN_PATH,
    allow_pickle=False
)

print()
print("X_train shape:")
print(X_train.shape)

print()
print("y_train shape:")
print(y_train.shape)

# ======================================================================
# 7. LOAD VALIDATION SEQUENCES
# ======================================================================

print()
print("=" * 70)
print("LOADING VALIDATION SEQUENCES")
print("=" * 70)

X_validation = np.load(
    X_VALIDATION_PATH,
    allow_pickle=False
)

y_validation = np.load(
    Y_VALIDATION_PATH,
    allow_pickle=False
)

print()
print("X_validation shape:")
print(X_validation.shape)

print()
print("y_validation shape:")
print(y_validation.shape)

# ======================================================================
# 8. INPUT SHAPE CHECK
# ======================================================================

print()
print("=" * 70)
print("INPUT SHAPE CHECK")
print("=" * 70)

expected_shape = (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
)

print(
    "Expected X shape per sample:",
    expected_shape
)

print(
    "Actual X_train shape per sample:",
    X_train.shape[1:]
)

print(
    "Actual X_validation shape per sample:",
    X_validation.shape[1:]
)

if X_train.shape[1:] != expected_shape:

    raise ValueError(
        "Training sequence shape does not match "
        "TCN architecture."
    )

if X_validation.shape[1:] != expected_shape:

    raise ValueError(
        "Validation sequence shape does not match "
        "TCN architecture."
    )

if len(X_train) != len(y_train):

    raise ValueError(
        "Training X/y sequence counts do not match."
    )

if len(X_validation) != len(y_validation):

    raise ValueError(
        "Validation X/y sequence counts do not match."
    )

print()
print("✓ Training input shape verified.")
print("✓ Validation input shape verified.")
print("✓ Training X/y counts verified.")
print("✓ Validation X/y counts verified.")

# ======================================================================
# 9. DATA SAFETY CHECK
# ======================================================================

print()
print("=" * 70)
print("TRAINING DATA SAFETY CHECK")
print("=" * 70)

print()
print("Training X NaN:")
print(np.isnan(X_train).sum())

print(
    "Training X infinite:"
)
print(
    np.isinf(X_train).sum()
)

print()
print("Training y NaN:")
print(np.isnan(y_train).sum())

print(
    "Training y infinite:"
)
print(
    np.isinf(y_train).sum()
)

print()
print("Validation X NaN:")
print(np.isnan(X_validation).sum())

print(
    "Validation X infinite:"
)
print(
    np.isinf(X_validation).sum()
)

print()
print("Validation y NaN:")
print(np.isnan(y_validation).sum())

print(
    "Validation y infinite:"
)
print(
    np.isinf(y_validation).sum()
)

if np.isnan(X_train).any():
    raise ValueError(
        "NaN detected in training X."
    )

if np.isinf(X_train).any():
    raise ValueError(
        "Infinite value detected in training X."
    )

if np.isnan(y_train).any():
    raise ValueError(
        "NaN detected in training target."
    )

if np.isinf(y_train).any():
    raise ValueError(
        "Infinite value detected in training target."
    )

if np.isnan(X_validation).any():
    raise ValueError(
        "NaN detected in validation X."
    )

if np.isinf(X_validation).any():
    raise ValueError(
        "Infinite value detected in validation X."
    )

if np.isnan(y_validation).any():
    raise ValueError(
        "NaN detected in validation target."
    )

if np.isinf(y_validation).any():
    raise ValueError(
        "Infinite value detected in validation target."
    )

print()
print("✓ Training data passed safety checks.")
print("✓ Validation data passed safety checks.")

# ======================================================================
# 10. LOAD TCN ARCHITECTURE
# ======================================================================

print()
print("=" * 70)
print("LOADING TCN ARCHITECTURE")
print("=" * 70)

model = load_model(
    ARCHITECTURE_PATH,
    compile=False
)

print()
print("Model type:")
print(type(model))

print()
print("Model name:")
print(model.name)

print()
print("Model input shape:")
print(model.input_shape)

print()
print("Model output shape:")
print(model.output_shape)

if model.input_shape[1:] != (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
):

    raise ValueError(
        "TCN model input shape is incorrect."
    )

if model.output_shape[-1] != 1:

    raise ValueError(
        "TCN model output shape is incorrect."
    )

print()
print("✓ TCN architecture loaded.")
print("✓ Input shape verified.")
print("✓ Output shape verified.")

# ======================================================================
# 11. ARCHITECTURE SUMMARY
# ======================================================================

print()
print("=" * 70)
print("TCN ARCHITECTURE")
print("=" * 70)

model.summary()

print()
print("Total parameters:")
print(model.count_params())

# ======================================================================
# 12. COMPILE MODEL
# ======================================================================

print()
print("=" * 70)
print("COMPILING TCN")
print("=" * 70)

optimizer = Adam(
    learning_rate=LEARNING_RATE
)

loss_function = Huber(
    delta=HUBER_DELTA
)

model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

print()
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)

print("Loss: Huber")
print("Huber delta:", HUBER_DELTA)

print("Metric: MAE")

print()
print("✓ Model compiled.")

# ======================================================================
# 13. CALLBACK PATHS
# ======================================================================

BEST_MODEL_PATH = (
    MODEL_DIR /
    "tcn_best_model.keras"
)

FINAL_MODEL_PATH = (
    MODEL_DIR /
    "tcn_final_model.keras"
)

HISTORY_PATH = (
    EVALUATION_DIR /
    "tcn_training_history.csv"
)

CONFIGURATION_PATH = (
    EVALUATION_DIR /
    "tcn_training_configuration.json"
)

# ======================================================================
# 14. CALLBACKS
# ======================================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=REDUCE_LR_FACTOR,
    patience=REDUCE_LR_PATIENCE,
    min_lr=MIN_LR,
    mode="min",
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=str(BEST_MODEL_PATH),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

callbacks = [
    early_stopping,
    reduce_lr,
    checkpoint
]

print()
print("=" * 70)
print("TRAINING CALLBACKS")
print("=" * 70)

print()
print("EarlyStopping:")
print("Monitor: val_loss")
print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)
print("Restore best weights: True")

print()
print("ReduceLROnPlateau:")
print("Monitor: val_loss")
print(
    "Factor:",
    REDUCE_LR_FACTOR
)
print(
    "Patience:",
    REDUCE_LR_PATIENCE
)

print()
print("ModelCheckpoint:")
print("Monitor: val_loss")
print("Save best only: True")

# ======================================================================
# 15. SAFETY — TEST DATA
# ======================================================================

print()
print("=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset loaded: False")
print("Test sequences loaded: False")
print("Test predictions generated: False")

print()
print(
    "✓ Test data is NOT part of this training process."
)

# ======================================================================
# 16. TRAIN
# ======================================================================

print()
print("=" * 70)
print("STARTING TCN TRAINING")
print("=" * 70)

print()
print("Training samples:")
print(len(X_train))

print()
print("Validation samples:")
print(len(X_validation))

print()
print("Batch size:")
print(BATCH_SIZE)

print()
print("Maximum epochs:")
print(MAX_EPOCHS)

print()
print("Starting training...")

start_time = time.time()

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_validation,
        y_validation
    ),

    epochs=MAX_EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    shuffle=True,

    verbose=1
)

training_time = (
    time.time() -
    start_time
)

print()
print("=" * 70)
print("TCN TRAINING COMPLETE")
print("=" * 70)

print()
print(
    "Training time:",
    f"{training_time:.2f} seconds"
)

print()
print(
    "Epochs completed:",
    len(history.history["loss"])
)

# ======================================================================
# 17. TRAINING HISTORY
# ======================================================================

history_df = pd.DataFrame(
    history.history
)

history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(history_df) + 1
    )
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print()
print("=" * 70)
print("TRAINING HISTORY")
print("=" * 70)

print()
print(history_df.to_string(index=False))

# ======================================================================
# 18. BEST EPOCH
# ======================================================================

best_epoch_index = int(
    np.argmin(
        history.history["val_loss"]
    )
)

best_epoch = (
    best_epoch_index + 1
)

best_val_loss = float(
    history.history["val_loss"][
        best_epoch_index
    ]
)

best_train_loss = float(
    history.history["loss"][
        best_epoch_index
    ]
)

print()
print("=" * 70)
print("BEST VALIDATION EPOCH")
print("=" * 70)

print()
print("Best epoch:")
print(best_epoch)

print()
print("Best training loss:")
print(best_train_loss)

print()
print("Best validation loss:")
print(best_val_loss)

# ======================================================================
# 19. SAVE FINAL MODEL
# ======================================================================

print()
print("=" * 70)
print("SAVING FINAL TCN MODEL")
print("=" * 70)

model.save(
    FINAL_MODEL_PATH
)

print()
print("Best checkpoint:")
print(BEST_MODEL_PATH)

print()
print("Final model:")
print(FINAL_MODEL_PATH)

# ======================================================================
# 20. TRAINING CONFIGURATION
# ======================================================================

training_configuration = {

    "model":
        "TCN_CHL_Forecaster",

    "random_seed":
        SEED,

    "sequence_length":
        SEQUENCE_LENGTH,

    "number_of_features":
        NUMBER_FEATURES,

    "input_shape":
        [
            SEQUENCE_LENGTH,
            NUMBER_FEATURES
        ],

    "target":
        TARGET,

    "batch_size":
        BATCH_SIZE,

    "maximum_epochs":
        MAX_EPOCHS,

    "epochs_completed":
        len(history.history["loss"]),

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "loss":
        "Huber",

    "huber_delta":
        HUBER_DELTA,

    "early_stopping":
        True,

    "early_stopping_monitor":
        "val_loss",

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "restore_best_weights":
        True,

    "reduce_lr_on_plateau":
        True,

    "reduce_lr_monitor":
        "val_loss",

    "reduce_lr_factor":
        REDUCE_LR_FACTOR,

    "reduce_lr_patience":
        REDUCE_LR_PATIENCE,

    "minimum_learning_rate":
        MIN_LR,

    "model_checkpoint":
        True,

    "best_epoch":
        best_epoch,

    "best_validation_loss":
        best_val_loss,

    "best_training_loss":
        best_train_loss,

    "training_samples":
        int(len(X_train)),

    "validation_samples":
        int(len(X_validation)),

    "training_time_seconds":
        float(training_time),

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "test_sequences_loaded":
        False,

    "model_retrained":
        True
}

with open(
    CONFIGURATION_PATH,
    "w"
) as file:

    json.dump(
        training_configuration,
        file,
        indent=4
    )

print()
print(
    "Training configuration saved:"
)

print(
    CONFIGURATION_PATH
)

# ======================================================================
# 21. BEST MODEL RELOAD CHECK
# ======================================================================

print()
print("=" * 70)
print("BEST MODEL VERIFICATION")
print("=" * 70)

best_model = load_model(
    BEST_MODEL_PATH,
    compile=False
)

print()
print("Best model loaded:")
print(BEST_MODEL_PATH)

print()
print("Input shape:")
print(best_model.input_shape)

print()
print("Output shape:")
print(best_model.output_shape)

print()
print("Parameters:")
print(best_model.count_params())

if best_model.input_shape[1:] != (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
):

    raise ValueError(
        "Saved best TCN model has incorrect input shape."
    )

if best_model.output_shape[-1] != 1:

    raise ValueError(
        "Saved best TCN model has incorrect output shape."
    )

print()
print("✓ Best TCN model successfully verified.")

# ======================================================================
# 22. FINAL SAFETY CHECK
# ======================================================================

print()
print("=" * 70)
print("FINAL 31.9P-16K SAFETY CHECK")
print("=" * 70)

print()
print("Training performed: True")

print(
    "Validation data used: True"
)

print(
    "Test data loaded: False"
)

print(
    "Test data used: False"
)

print(
    "Existing RF modified: False"
)

print(
    "Existing XGBoost modified: False"
)

print(
    "Existing CNN-LSTM modified: False"
)

print(
    "TCN architecture modified: False"
)

print()
print(
    "✓ TCN trained using training data."
)

print(
    "✓ Validation data used only for monitoring "
    "and model selection."
)

print(
    "✓ Best model selected using validation loss."
)

print(
    "✓ Test data was NOT loaded."
)

print(
    "✓ Test data was NOT used."
)

print(
    "✓ Existing models remain untouched."
)

# ======================================================================
# 23. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("STEP 31.9P-16K COMPLETE")
print("=" * 70)

print()
print("🏆 TCN TRAINING COMPLETED")

print()
print("Training sequences:")
print(len(X_train))

print()
print("Validation sequences:")
print(len(X_validation))

print()
print("Input shape:")
print(
    (
        SEQUENCE_LENGTH,
        NUMBER_FEATURES
    )
)

print()
print("Parameters:")
print(
    best_model.count_params()
)

print()
print("Best epoch:")
print(best_epoch)

print()
print("Best validation loss:")
print(best_val_loss)

print()
print("Training time:")
print(
    f"{training_time:.2f} seconds"
)

print()
print("Best model:")
print(BEST_MODEL_PATH)

print()
print("Training history:")
print(HISTORY_PATH)

print()
print("Training configuration:")
print(CONFIGURATION_PATH)

print()
print("TEST DATA:")
print("NOT USED")

print()
print("=" * 70)
print("NEXT STEP: 31.9P-16L — TCN VALIDATION EVALUATION")
print("=" * 70)

STEP 31.9P-16K
TCN TRAINING

REPRODUCIBILITY
Random seed: 42

GPU VERIFICATION
Number of GPUs detected: 1
GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ TensorFlow GPU available.

PROJECT PATHS

TCN directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN

Sequence directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/sequences

Model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/models

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/evaluation

TCN TRAINING CONFIGURATION
Sequence length: 7
Number of features: 69
Input shape: (7, 69)
Target: chlorophyll_a_log1p
Batch size: 256
Maximum epochs: 50
Learning rate: 0.001
Loss: Huber
Huber delta: 1.0
Early stopping patience: 8
ReduceLR patience: 4
ReduceLR factor: 0.5
Minimum learning rate: 1e-06

REQUIRED FILE CHECK

Training X:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/seq

FileNotFoundError: Required file not found: /content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/sequences/tcn_X_training.npy

In [2]:
# ======================================================================
# STEP 31.9P-16K
# TCN TRAINING
#
# IMPORTANT:
# - Uses ONLY training and validation sequences
# - Test data is NOT loaded
# - Existing models are NOT modified
# - TCN architecture from 31.9P-16I is preserved
# ======================================================================

import os
import gc
import json
import random
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("=" * 70)
print("STEP 31.9P-16K")
print("TCN TRAINING")
print("=" * 70)

# ======================================================================
# 1. REPRODUCIBILITY
# ======================================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print()
print("=" * 70)
print("REPRODUCIBILITY")
print("=" * 70)

print("Random seed:", SEED)

# ======================================================================
# 2. GPU VERIFICATION
# ======================================================================

print()
print("=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

gpus = tf.config.list_physical_devices("GPU")

print(
    "Number of GPUs detected:",
    len(gpus)
)

if len(gpus) > 0:

    for gpu in gpus:
        print("GPU:", gpu)

    print()
    print("✓ TensorFlow GPU available.")

else:

    print()
    print("⚠ No GPU detected.")
    print("Training will use CPU.")

# ======================================================================
# 3. PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

TCN_DIR = (
    PROJECT_DIR /
    "06_models" /
    "05_TCN"
)

SEQUENCE_DIR = (
    TCN_DIR /
    "sequences"
)

MODEL_DIR = (
    TCN_DIR /
    "models"
)

EVALUATION_DIR = (
    TCN_DIR /
    "evaluation"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)

print()
print("TCN directory:")
print(TCN_DIR)

print()
print("Sequence directory:")
print(SEQUENCE_DIR)

print()
print("Model directory:")
print(MODEL_DIR)

print()
print("Evaluation directory:")
print(EVALUATION_DIR)

# ======================================================================
# 4. TRAINING CONFIGURATION
# ======================================================================

SEQUENCE_LENGTH = 7
NUMBER_FEATURES = 69

TARGET = "chlorophyll_a_log1p"

BATCH_SIZE = 256
MAX_EPOCHS = 50

LEARNING_RATE = 0.001

HUBER_DELTA = 1.0

EARLY_STOPPING_PATIENCE = 8

REDUCE_LR_PATIENCE = 4
REDUCE_LR_FACTOR = 0.5

MIN_LR = 1e-6

print()
print("=" * 70)
print("TCN TRAINING CONFIGURATION")
print("=" * 70)

print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", NUMBER_FEATURES)
print("Input shape:", (SEQUENCE_LENGTH, NUMBER_FEATURES))

print("Target:", TARGET)

print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)

print("Learning rate:", LEARNING_RATE)

print("Loss: Huber")
print("Huber delta:", HUBER_DELTA)

print(
    "Early stopping patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "ReduceLR patience:",
    REDUCE_LR_PATIENCE
)

print(
    "ReduceLR factor:",
    REDUCE_LR_FACTOR
)

print("Minimum learning rate:", MIN_LR)

# ======================================================================
# 5. REQUIRED FILES
# ======================================================================

X_TRAIN_PATH = (
    SEQUENCE_DIR /
    "tcn_X_training.npy"
)

Y_TRAIN_PATH = (
    SEQUENCE_DIR /
    "tcn_y_log_training.npy"
)

X_VALIDATION_PATH = (
    SEQUENCE_DIR /
    "tcn_X_validation.npy"
)

Y_VALIDATION_PATH = (
    SEQUENCE_DIR /
    "tcn_y_log_validation.npy"
)

ARCHITECTURE_PATH = (
    TCN_DIR /
    "architectures" /
    "tcn_untrained.keras"
)

print()
print("=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

required_files = {

    "Training X":
        X_TRAIN_PATH,

    "Training y":
        Y_TRAIN_PATH,

    "Validation X":
        X_VALIDATION_PATH,

    "Validation y":
        Y_VALIDATION_PATH,

    "TCN architecture":
        ARCHITECTURE_PATH
}

for name, path in required_files.items():

    print()
    print(name + ":")
    print(path)

    print(
        "Exists:",
        path.exists()
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print()
print("✓ All required training files found.")

# ======================================================================
# 6. LOAD TRAINING SEQUENCES
# ======================================================================

print()
print("=" * 70)
print("LOADING TRAINING SEQUENCES")
print("=" * 70)

X_train = np.load(
    X_TRAIN_PATH,
    allow_pickle=False
)

y_train = np.load(
    Y_TRAIN_PATH,
    allow_pickle=False
)

print()
print("X_train shape:")
print(X_train.shape)

print()
print("y_train shape:")
print(y_train.shape)

# ======================================================================
# 7. LOAD VALIDATION SEQUENCES
# ======================================================================

print()
print("=" * 70)
print("LOADING VALIDATION SEQUENCES")
print("=" * 70)

X_validation = np.load(
    X_VALIDATION_PATH,
    allow_pickle=False
)

y_validation = np.load(
    Y_VALIDATION_PATH,
    allow_pickle=False
)

print()
print("X_validation shape:")
print(X_validation.shape)

print()
print("y_validation shape:")
print(y_validation.shape)

# ======================================================================
# 8. INPUT SHAPE CHECK
# ======================================================================

print()
print("=" * 70)
print("INPUT SHAPE CHECK")
print("=" * 70)

expected_shape = (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
)

print(
    "Expected X shape per sample:",
    expected_shape
)

print(
    "Actual X_train shape per sample:",
    X_train.shape[1:]
)

print(
    "Actual X_validation shape per sample:",
    X_validation.shape[1:]
)

if X_train.shape[1:] != expected_shape:

    raise ValueError(
        "Training sequence shape does not match "
        "TCN architecture."
    )

if X_validation.shape[1:] != expected_shape:

    raise ValueError(
        "Validation sequence shape does not match "
        "TCN architecture."
    )

if len(X_train) != len(y_train):

    raise ValueError(
        "Training X/y sequence counts do not match."
    )

if len(X_validation) != len(y_validation):

    raise ValueError(
        "Validation X/y sequence counts do not match."
    )

print()
print("✓ Training input shape verified.")
print("✓ Validation input shape verified.")
print("✓ Training X/y counts verified.")
print("✓ Validation X/y counts verified.")

# ======================================================================
# 9. DATA SAFETY CHECK
# ======================================================================

print()
print("=" * 70)
print("TRAINING DATA SAFETY CHECK")
print("=" * 70)

print()
print("Training X NaN:")
print(np.isnan(X_train).sum())

print(
    "Training X infinite:"
)
print(
    np.isinf(X_train).sum()
)

print()
print("Training y NaN:")
print(np.isnan(y_train).sum())

print(
    "Training y infinite:"
)
print(
    np.isinf(y_train).sum()
)

print()
print("Validation X NaN:")
print(np.isnan(X_validation).sum())

print(
    "Validation X infinite:"
)
print(
    np.isinf(X_validation).sum()
)

print()
print("Validation y NaN:")
print(np.isnan(y_validation).sum())

print(
    "Validation y infinite:"
)
print(
    np.isinf(y_validation).sum()
)

if np.isnan(X_train).any():
    raise ValueError(
        "NaN detected in training X."
    )

if np.isinf(X_train).any():
    raise ValueError(
        "Infinite value detected in training X."
    )

if np.isnan(y_train).any():
    raise ValueError(
        "NaN detected in training target."
    )

if np.isinf(y_train).any():
    raise ValueError(
        "Infinite value detected in training target."
    )

if np.isnan(X_validation).any():
    raise ValueError(
        "NaN detected in validation X."
    )

if np.isinf(X_validation).any():
    raise ValueError(
        "Infinite value detected in validation X."
    )

if np.isnan(y_validation).any():
    raise ValueError(
        "NaN detected in validation target."
    )

if np.isinf(y_validation).any():
    raise ValueError(
        "Infinite value detected in validation target."
    )

print()
print("✓ Training data passed safety checks.")
print("✓ Validation data passed safety checks.")

# ======================================================================
# 10. LOAD TCN ARCHITECTURE
# ======================================================================

print()
print("=" * 70)
print("LOADING TCN ARCHITECTURE")
print("=" * 70)

model = load_model(
    ARCHITECTURE_PATH,
    compile=False
)

print()
print("Model type:")
print(type(model))

print()
print("Model name:")
print(model.name)

print()
print("Model input shape:")
print(model.input_shape)

print()
print("Model output shape:")
print(model.output_shape)

if model.input_shape[1:] != (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
):

    raise ValueError(
        "TCN model input shape is incorrect."
    )

if model.output_shape[-1] != 1:

    raise ValueError(
        "TCN model output shape is incorrect."
    )

print()
print("✓ TCN architecture loaded.")
print("✓ Input shape verified.")
print("✓ Output shape verified.")

# ======================================================================
# 11. ARCHITECTURE SUMMARY
# ======================================================================

print()
print("=" * 70)
print("TCN ARCHITECTURE")
print("=" * 70)

model.summary()

print()
print("Total parameters:")
print(model.count_params())

# ======================================================================
# 12. COMPILE MODEL
# ======================================================================

print()
print("=" * 70)
print("COMPILING TCN")
print("=" * 70)

optimizer = Adam(
    learning_rate=LEARNING_RATE
)

loss_function = Huber(
    delta=HUBER_DELTA
)

model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

print()
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)

print("Loss: Huber")
print("Huber delta:", HUBER_DELTA)

print("Metric: MAE")

print()
print("✓ Model compiled.")

# ======================================================================
# 13. CALLBACK PATHS
# ======================================================================

BEST_MODEL_PATH = (
    MODEL_DIR /
    "tcn_best_model.keras"
)

FINAL_MODEL_PATH = (
    MODEL_DIR /
    "tcn_final_model.keras"
)

HISTORY_PATH = (
    EVALUATION_DIR /
    "tcn_training_history.csv"
)

CONFIGURATION_PATH = (
    EVALUATION_DIR /
    "tcn_training_configuration.json"
)

# ======================================================================
# 14. CALLBACKS
# ======================================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=REDUCE_LR_FACTOR,
    patience=REDUCE_LR_PATIENCE,
    min_lr=MIN_LR,
    mode="min",
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=str(BEST_MODEL_PATH),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

callbacks = [
    early_stopping,
    reduce_lr,
    checkpoint
]

print()
print("=" * 70)
print("TRAINING CALLBACKS")
print("=" * 70)

print()
print("EarlyStopping:")
print("Monitor: val_loss")
print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)
print("Restore best weights: True")

print()
print("ReduceLROnPlateau:")
print("Monitor: val_loss")
print(
    "Factor:",
    REDUCE_LR_FACTOR
)
print(
    "Patience:",
    REDUCE_LR_PATIENCE
)

print()
print("ModelCheckpoint:")
print("Monitor: val_loss")
print("Save best only: True")

# ======================================================================
# 15. SAFETY — TEST DATA
# ======================================================================

print()
print("=" * 70)
print("TEST DATA SAFETY")
print("=" * 70)

print("Test dataset loaded: False")
print("Test sequences loaded: False")
print("Test predictions generated: False")

print()
print(
    "✓ Test data is NOT part of this training process."
)

# ======================================================================
# 16. TRAIN
# ======================================================================

print()
print("=" * 70)
print("STARTING TCN TRAINING")
print("=" * 70)

print()
print("Training samples:")
print(len(X_train))

print()
print("Validation samples:")
print(len(X_validation))

print()
print("Batch size:")
print(BATCH_SIZE)

print()
print("Maximum epochs:")
print(MAX_EPOCHS)

print()
print("Starting training...")

start_time = time.time()

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_validation,
        y_validation
    ),

    epochs=MAX_EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    shuffle=True,

    verbose=1
)

training_time = (
    time.time() -
    start_time
)

print()
print("=" * 70)
print("TCN TRAINING COMPLETE")
print("=" * 70)

print()
print(
    "Training time:",
    f"{training_time:.2f} seconds"
)

print()
print(
    "Epochs completed:",
    len(history.history["loss"])
)

# ======================================================================
# 17. TRAINING HISTORY
# ======================================================================

history_df = pd.DataFrame(
    history.history
)

history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(history_df) + 1
    )
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

print()
print("=" * 70)
print("TRAINING HISTORY")
print("=" * 70)

print()
print(history_df.to_string(index=False))

# ======================================================================
# 18. BEST EPOCH
# ======================================================================

best_epoch_index = int(
    np.argmin(
        history.history["val_loss"]
    )
)

best_epoch = (
    best_epoch_index + 1
)

best_val_loss = float(
    history.history["val_loss"][
        best_epoch_index
    ]
)

best_train_loss = float(
    history.history["loss"][
        best_epoch_index
    ]
)

print()
print("=" * 70)
print("BEST VALIDATION EPOCH")
print("=" * 70)

print()
print("Best epoch:")
print(best_epoch)

print()
print("Best training loss:")
print(best_train_loss)

print()
print("Best validation loss:")
print(best_val_loss)

# ======================================================================
# 19. SAVE FINAL MODEL
# ======================================================================

print()
print("=" * 70)
print("SAVING FINAL TCN MODEL")
print("=" * 70)

model.save(
    FINAL_MODEL_PATH
)

print()
print("Best checkpoint:")
print(BEST_MODEL_PATH)

print()
print("Final model:")
print(FINAL_MODEL_PATH)

# ======================================================================
# 20. TRAINING CONFIGURATION
# ======================================================================

training_configuration = {

    "model":
        "TCN_CHL_Forecaster",

    "random_seed":
        SEED,

    "sequence_length":
        SEQUENCE_LENGTH,

    "number_of_features":
        NUMBER_FEATURES,

    "input_shape":
        [
            SEQUENCE_LENGTH,
            NUMBER_FEATURES
        ],

    "target":
        TARGET,

    "batch_size":
        BATCH_SIZE,

    "maximum_epochs":
        MAX_EPOCHS,

    "epochs_completed":
        len(history.history["loss"]),

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "loss":
        "Huber",

    "huber_delta":
        HUBER_DELTA,

    "early_stopping":
        True,

    "early_stopping_monitor":
        "val_loss",

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "restore_best_weights":
        True,

    "reduce_lr_on_plateau":
        True,

    "reduce_lr_monitor":
        "val_loss",

    "reduce_lr_factor":
        REDUCE_LR_FACTOR,

    "reduce_lr_patience":
        REDUCE_LR_PATIENCE,

    "minimum_learning_rate":
        MIN_LR,

    "model_checkpoint":
        True,

    "best_epoch":
        best_epoch,

    "best_validation_loss":
        best_val_loss,

    "best_training_loss":
        best_train_loss,

    "training_samples":
        int(len(X_train)),

    "validation_samples":
        int(len(X_validation)),

    "training_time_seconds":
        float(training_time),

    "test_data_loaded":
        False,

    "test_data_used":
        False,

    "test_sequences_loaded":
        False,

    "model_retrained":
        True
}

with open(
    CONFIGURATION_PATH,
    "w"
) as file:

    json.dump(
        training_configuration,
        file,
        indent=4
    )

print()
print(
    "Training configuration saved:"
)

print(
    CONFIGURATION_PATH
)

# ======================================================================
# 21. BEST MODEL RELOAD CHECK
# ======================================================================

print()
print("=" * 70)
print("BEST MODEL VERIFICATION")
print("=" * 70)

best_model = load_model(
    BEST_MODEL_PATH,
    compile=False
)

print()
print("Best model loaded:")
print(BEST_MODEL_PATH)

print()
print("Input shape:")
print(best_model.input_shape)

print()
print("Output shape:")
print(best_model.output_shape)

print()
print("Parameters:")
print(best_model.count_params())

if best_model.input_shape[1:] != (
    SEQUENCE_LENGTH,
    NUMBER_FEATURES
):

    raise ValueError(
        "Saved best TCN model has incorrect input shape."
    )

if best_model.output_shape[-1] != 1:

    raise ValueError(
        "Saved best TCN model has incorrect output shape."
    )

print()
print("✓ Best TCN model successfully verified.")

# ======================================================================
# 22. FINAL SAFETY CHECK
# ======================================================================

print()
print("=" * 70)
print("FINAL 31.9P-16K SAFETY CHECK")
print("=" * 70)

print()
print("Training performed: True")

print(
    "Validation data used: True"
)

print(
    "Test data loaded: False"
)

print(
    "Test data used: False"
)

print(
    "Existing RF modified: False"
)

print(
    "Existing XGBoost modified: False"
)

print(
    "Existing CNN-LSTM modified: False"
)

print(
    "TCN architecture modified: False"
)

print()
print(
    "✓ TCN trained using training data."
)

print(
    "✓ Validation data used only for monitoring "
    "and model selection."
)

print(
    "✓ Best model selected using validation loss."
)

print(
    "✓ Test data was NOT loaded."
)

print(
    "✓ Test data was NOT used."
)

print(
    "✓ Existing models remain untouched."
)

# ======================================================================
# 23. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("STEP 31.9P-16K COMPLETE")
print("=" * 70)

print()
print("🏆 TCN TRAINING COMPLETED")

print()
print("Training sequences:")
print(len(X_train))

print()
print("Validation sequences:")
print(len(X_validation))

print()
print("Input shape:")
print(
    (
        SEQUENCE_LENGTH,
        NUMBER_FEATURES
    )
)

print()
print("Parameters:")
print(
    best_model.count_params()
)

print()
print("Best epoch:")
print(best_epoch)

print()
print("Best validation loss:")
print(best_val_loss)

print()
print("Training time:")
print(
    f"{training_time:.2f} seconds"
)

print()
print("Best model:")
print(BEST_MODEL_PATH)

print()
print("Training history:")
print(HISTORY_PATH)

print()
print("Training configuration:")
print(CONFIGURATION_PATH)

print()
print("TEST DATA:")
print("NOT USED")

print()
print("=" * 70)
print("NEXT STEP: 31.9P-16L — TCN VALIDATION EVALUATION")
print("=" * 70)

STEP 31.9P-16K
TCN TRAINING

REPRODUCIBILITY
Random seed: 42

GPU VERIFICATION
Number of GPUs detected: 1
GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ TensorFlow GPU available.

PROJECT PATHS

TCN directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN

Sequence directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/sequences

Model directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/models

Evaluation directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/evaluation

TCN TRAINING CONFIGURATION
Sequence length: 7
Number of features: 69
Input shape: (7, 69)
Target: chlorophyll_a_log1p
Batch size: 256
Maximum epochs: 50
Learning rate: 0.001
Loss: Huber
Huber delta: 1.0
Early stopping patience: 8
ReduceLR patience: 4
ReduceLR factor: 0.5
Minimum learning rate: 1e-06

REQUIRED FILE CHECK

Training X:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/seq

Model: "TCN_CHL_Forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tcn_input           │ (None, 7, 69)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_conv    │ (None, 7, 64)     │     13,312 │ tcn_input[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_norm    │ (None, 7, 64)     │        128 │ tcn_block_1_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_relu    │ (None, 7, 64)     │          0 │ tcn_block_1_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_residu… │ (None, 7, 64)     │      4,480 │ tcn_input[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_dropout │ (None, 7, 64)     │          0 │ tcn_block_1_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_1_add     │ (None, 7, 64)     │          0 │ tcn_block_1_resi… │
│ (Add)               │                   │            │ tcn_block_1_drop… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_conv    │ (None, 7, 64)     │     12,352 │ tcn_block_1_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_norm    │ (None, 7, 64)     │        128 │ tcn_block_2_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_relu    │ (None, 7, 64)     │          0 │ tcn_block_2_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_dropout │ (None, 7, 64)     │          0 │ tcn_block_2_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_2_add     │ (None, 7, 64)     │          0 │ tcn_block_1_add[… │
│ (Add)               │                   │            │ tcn_block_2_drop… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_conv    │ (None, 7, 64)     │     12,352 │ tcn_block_2_add[… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_norm    │ (None, 7, 64)     │        128 │ tcn_block_3_conv… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_relu    │ (None, 7, 64)     │          0 │ tcn_block_3_norm… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_dropout │ (None, 7, 64)     │          0 │ tcn_block_3_relu… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn_block_3_add     │ (None, 7, 64)     │          0 │ tcn_block_2_add[

 Total params: 44,993 (175.75 KB)

 Trainable params: 44,993 (175.75 KB)

 Non-trainable params: 0 (0.00 B)


Total parameters:
44993

COMPILING TCN

Optimizer: Adam
Learning rate: 0.001
Loss: Huber
Huber delta: 1.0
Metric: MAE

✓ Model compiled.

TRAINING CALLBACKS

EarlyStopping:
Monitor: val_loss
Patience: 8
Restore best weights: True

ReduceLROnPlateau:
Monitor: val_loss
Factor: 0.5
Patience: 4

ModelCheckpoint:
Monitor: val_loss
Save best only: True

TEST DATA SAFETY
Test dataset loaded: False
Test sequences loaded: False
Test predictions generated: False

✓ Test data is NOT part of this training process.

STARTING TCN TRAINING

Training samples:
729912

Validation samples:
157182

Batch size:
256

Maximum epochs:
50

Starting training...
Epoch 1/50
2852/2852 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0243 - mae: 0.1220
Epoch 1: val_loss improved from None to 0.01996, saving model to /content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/models/tcn_best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/05_TCN/models/tcn